In [ ]:
import json
from typing import Optional
import requests
import uuid

In [1]:
from pathlib import Path

story_path = Path("outputs") / f"story_{1}.md"

In [2]:
story_path

PosixPath('outputs/story_1.md')

In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()

OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
OPEN_ROUTER_COMPLETION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
OPEN_ROUTER_URL = "https://openrouter.ai/api/v1"

OLLAMA_COMPLETION_MODEL = (
# "qwen3:latest"
"gemma4:e4b"
)


OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
OLLAMA_API_KEY='ollama'


OPEN_ROUTER_HEADERS = {
        "HTTP-Referer": "https://github.com/pixelbuildlab/be-story-teller",
        "X-OpenRouter-Title": "Be story teller - Agentic way to stories",
    }

WORKER_API_KEY = os.getenv("WORKER_API_KEY") 
WORKER_API_URL = os.getenv("WORKER_API_URL")

In [ ]:
USE_OLLAMA=True
MODEL= OLLAMA_COMPLETION_MODEL if USE_OLLAMA else OPEN_ROUTER_COMPLETION_MODEL
API_URL= OLLAMA_API_URL if USE_OLLAMA else OPEN_ROUTER_URL
API_KEY = OLLAMA_API_KEY if USE_OLLAMA else OPEN_ROUTER_API_KEY
HEADERS = None if USE_OLLAMA else OPEN_ROUTER_HEADERS

In [ ]:
USE_OLLAMA, MODEL, API_URL,  HEADERS

In [ ]:
import openai
from openai import OpenAI

client = OpenAI(
    base_url=API_URL,
    api_key=API_KEY,
    default_headers=HEADERS,
)


In [ ]:
SYSTEM_META_PROMPT = """
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Stories should be calming.
- Never include violence or horror.
- Keep language simple.
- If you need to use a tool, use it.
- Stories should be relaxing, pleasing to hear and lesson full.
- Once the story is written you need to work on the scene extraction and a prompt generation for image generation tool.
"""

In [ ]:
def AI(messages: list, tools: Optional[list] | None):
    chat_completion = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    return chat_completion

In [ ]:
META_PROMPT_OPTIMIZER = """
You are an expert prompt optimizer for children's story generation.

Your task is to transform short or incomplete user requests into rich, detailed prompts for a story-writing AI.

Rules:
- Preserve the user's original intent.
- Add reasonable assumptions when details are missing.
- Specify:
  - protagonist
  - setting
  - conflict
  - tone
  - target age
  - ending
  - approximate length
- Do not write the story.
- Return ONLY the optimized prompt.
"""


async def story_prompt_optimizer_tool(prompt: str):
    print("starting story_prompt_optimizer_tool")
    messages = [
        {"role": "system", "content": META_PROMPT_OPTIMIZER},
        {
            "role": "user",
            "content": f"{prompt}",
        },
    ]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("story_prompt_optimizer_tool called")
    return response_message, chat_completion

In [ ]:
META_PROMPT_PLANNER = """
You are a professional children's story planner.

Your responsibility is to create a structured plan for a children's bedtime story.

The final story will be written by another AI, so only create the plan.

Story Requirements:
- Audience: Children aged 4–9.
- Genre: Bedtime story.
- Tone: Calm, gentle, comforting and relaxing.
- Language should remain simple.
- The story must teach a positive lesson or moral.
- Never include violence, horror, frightening scenes, or inappropriate themes.
- The story should be enjoyable to listen to before bedtime.

Generate ONLY valid JSON using the following structure:

{
    "title": "",
    "genre": "Children's Bedtime",
    "tone": "",
    "premise": "",
    "lesson": "",
    "ending": "",
    "characters": [
        {
            "name": "",
            "role": "",
            "personality": ""
        }
    ],
    "outline": [
        {
            "id": 1,
            "summary": "",
            "goal": ""
        }
    ]
}

Planning Guidelines:
- Create 3–5 outline sections.
- Introduce the main character in the first section.
- Present a small, age-appropriate challenge.
- Resolve the challenge peacefully.
- End with a satisfying, comforting conclusion.
- Ensure the lesson naturally emerges from the story.
- Keep every event suitable for children aged 4–9.
- If the user provides story details, preserve them.
- Otherwise, invent wholesome and creative details.

Rules:
- Return ONLY valid JSON.
- Do not write the actual story.
- Do not explain your reasoning.
- Do not include markdown.
"""

async def story_outline_planner_tool(optimized_prompt:str):
    # create an outline based on the system
    # this will help AI to be aligned with a single path
    # donot add characters and stray time waste
    print("Started story_outline_planner_tool")
    messages = [
        {"role": "system", "content": META_PROMPT_PLANNER},
        {
            "role": "user",
            "content": f"{optimized_prompt}",
        },
    ]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("End story_outline_planner_tool")
    return response_message, chat_completion

In [ ]:
META_PROMPT_WRITER = """
You are a professional children's story writer.

Write a complete bedtime story from the provided story plan.

Requirements:
- Follow the plan faithfully.
- Do not change the plot or lesson.
- Keep the language suitable for ages 4–9.
- Use a calm, soothing tone.
- Add natural dialogue where appropriate.
- Make the story engaging but relaxing.
- End naturally with the intended lesson.
- Return only the story.
"""


async def story_writer_tool(planned_story_json: dict):
    print("Started story_writer_tool")

    messages = [
    {
        "role": "system",
        "content": META_PROMPT_WRITER,
    },
    {
        "role": "user",
        "content": (
            "Write a complete children's bedtime story from the following "
            "structured story plan.\n\n"
            f"{json.dumps(planned_story_json, indent=2)}"
        ),
    },
]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("End story_writer_tool")

    return response_message, chat_completion

In [ ]:
# SceneExtractionTool

META_PROMPT_EXTRACTOR = """
You are a professional prompt engineer.
Your job is to extract some details from the information below.
These details are from a bed time story. It might be the actual story as well.
- You need to process details or story such that you return an optimized prompt for image generation.
- The image generation actual requires a minimal prompt to create image.
- Don't add much details to prompt as currently I only need a minimal image as scenery in story.
"""


async def story_scene_exactor_tool(created_story: str):
    print("Started story_scene_exactor_tool")

    messages = [
    {
        "role": "system",
        "content": META_PROMPT_EXTRACTOR,
    },
    {
        "role": "user",
        "content": (
            "Extract a scene from the following story and generate a prompt for making an image and using inside story."
            f"{created_story}"
        ),
    },
]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("End story_scene_exactor_tool")

    return response_message, chat_completion

In [ ]:
WORKER_API_KEY

In [ ]:
import requests, json


async def image_generation_tool(prompt: str):
    print("Started image_generation_tool")

    response = requests.post(
        WORKER_API_URL,
        json={"prompt": prompt,"model":"@cf/stabilityai/stable-diffusion-xl-base-1.0"},
        headers={"Authorization": f"Bearer {WORKER_API_KEY}"},
    )
    response.raise_for_status()
    if response.ok:
    # 3. Open a file in write-binary mode ('wb') and write the content
        image_id = uuid.uuid4()
        image_path = f"./outputs/{image_id}.png"
        print("Ending image_generation_tool")
        with open(image_path, "wb") as file:
            file.write(response.content)
            return f"./${image_id}.png", None
            

In [ ]:
# MarkdownWriterTool

# async def markdown_writer_tool():
    

META_PROMPT_FILE_WRITER = """
You will be given a story in almost md format and image for presenting the story.
- You need to just work on story formatting for md file only.
- Never rewrite story or improve any thing.
- Your main purpose is to return a formatted story with image embedded in md format.
- Return formatted markdown
- use <br> tag for newline.
- file will be available locally so need to work on adding as a pic only.
"""


async def markdown_writer_tool(created_story: str, generated_image_path:str):
    print("Started markdown_writer_tool")

    messages = [
    {
        "role": "system",
        "content": META_PROMPT_FILE_WRITER,
    },
    {
        "role": "user",
        "content": (
            "Story:"
            f"{created_story}"
            f"image path: {generated_image_path}"
        ),
    },
]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message
    
    formatted_story_markdown = response_message.content
    
    story_id  = uuid.uuid4()
    story_path = f"./outputs/story_{story_id}.md"
    with open(story_path, "w", encoding="utf-8") as f:
        f.write(formatted_story_markdown)
    
    print("End markdown_writer_tool")
    return response_message,  chat_completion


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "story_prompt_optimizer_tool",
            "description": "Optimize user input prompt to a level it creates stunning storyline. Not required if input prompt is valid and meaningful for a story",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "User input prompt to optimize",
                    }
                },
                "required": ["prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "story_outline_planner_tool",
            "description": "Work on storyline, scenes and characters for the story. It outputs a formatted JSON for storyline planning.",
            "parameters": {
                "type": "object",
                "properties": {
                    "optimized_prompt": {
                        "type": "string",
                        "description": "Optimized prompt to draft a storyline",
                    }
                },
                "required": ["optimized_prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "story_writer_tool",
            "description": "Writes the final children's story from the structured story plan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "planned_story_json": {
                        "type": "object",
                        "description": "Structured story plan produced by the story planner.",
                        "properties": {
                            "title": {"type": "string"},
                            "characters": {
                                "type": "array",
                                "items": {"type": "string"},
                            },
                            "setting": {"type": "string"},
                            "plot": {"type": "array", "items": {"type": "string"}},
                            "lesson": {"type": "string"},
                            "tone": {"type": "string"},
                        },
                        "required": ["characters", "setting", "plot", "lesson"],
                    }
                },
                "required": ["planned_story_json"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "story_scene_exactor_tool",
            "description": "Extracts and curates a prompt for image generation from the final created story",
            "parameters": {
                "type": "object",
                "properties": {
                    "created_story": {
                        "type": "string",
                        "description": "Finalized bed time story.",
                    }
                },
                "required": ["created_story"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "image_generation_tool",
            "description": "Generates and stores an image related to story using input prompt",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "Prompt to generate images based on.",
                    }
                },
                "required": ["prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "markdown_writer_tool",
            "description": "Final step in agentic story writer. Stories generated story and image in an output file in markdown format",
            "parameters": {
                "type": "object",
                "properties": {
                    "created_story": {
                        "type": "string",
                        "description": "Finalized story",
                    },
                    "generated_image_path": {
                        "type": "string",
                        "description": "Image path for illustration in story.",
                    },
                },
                "required": ["created_story", "generated_image_path"],
            },
        },
    },
]

In [ ]:
class AgentService:
    MESSAGES_LIST = []

    async def flow(self, prompt: str, META_PROMPT: str):
        try:
            messages = [
                {"role": "system", "content": META_PROMPT},
                {
                    "role": "user",
                    "content": f"{prompt}",
                },
            ]
            self.MESSAGES_LIST.extend(messages)

            while True:
                print("STARTING AGENT")
                chat_completion = AI(self.MESSAGES_LIST, tools)
                response_message = chat_completion.choices[0].message

                self.MESSAGES_LIST.append(response_message.model_dump())
                print(f"MAIN chat output: {response_message}")

                # If LLM returned tool calls, process them
                if (
                    hasattr(response_message, "tool_calls")
                    and response_message.tool_calls
                ):
                    try:
                        for tool_call in response_message.tool_calls:
                            function_name = tool_call.function.name
                            function_args = json.loads(tool_call.function.arguments)

                            print(f"Tool call: {function_name}, args: {function_args}")
                            agent_args = []

                            tool_function = globals()[function_name]

                            tool_result, tool_chat_completion = await tool_function(
                                *agent_args, **function_args
                            )

                            if hasattr(tool_result, "content"):
                                self.MESSAGES_LIST.append(
                                    {
                                        "role": "tool",
                                        "tool_name": function_name,
                                        "tool_call_id": tool_call.id,
                                        "content": tool_result.content,
                                    }
                                )
                            else:
                                self.MESSAGES_LIST.append(
                                    {
                                        "role": "tool",
                                        "tool_name": function_name,
                                        "tool_call_id": tool_call.id,
                                        "content": tool_result,
                                    }
                                )
                    except Exception as e:
                        print("got an exception while invoking tool")
                        # maybe handle these exception by tools itself?

                else:
                    # LIKELY TO STOP LOOP
                    # IF BUGGY USE A TOOL TO SOP THE LOOP.
                    return chat_completion

                # return chat_completion

        except openai.APIConnectionError as e:
            print(f"Network connectivity issue: {e}")
        except openai.RateLimitError as e:
            print(f"Rate limits hit or out of funds: {e}")
        except openai.APIStatusError as e:
            print(f"HTTP Error received (Status: {e.status_code}): {e.response}")

In [ ]:
agent =  AgentService()


In [ ]:
await agent.flow('parrot', SYSTEM_META_PROMPT)

In [ ]:
agent.MESSAGES_LIST